In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port', '0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
base_rdd = spark.sparkContext.textFile("/public/trendytech/orders/orders.csv")

In [ ]:
# input will be like this to next code 


1,2013-07-25 00:00:00.0,11599,CLOSED
2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
3,2013-07-25 00:00:00.0,12111,COMPLETE
4,2013-07-25 00:00:00.0,8827,CLOSED
5,2013-07-25 00:00:00.0,11318,COMPLETE

In [3]:
#  x.split(",")[2])  is customer id 
# (x.split(",")[3] is order_status 
mapped_rdd = base_rdd.map(lambda x:(x.split(",")[3], x.split(",")[2]))

# the output will be like this here 

In [ ]:
(CLOSED,11599)
(PENDING_PAYMENT,256)
(COMPLETE,12111)
(CLOSED,11502)

In [4]:
# we apply groupByKey here 
# here all the closed will go to one machine and the result will be shown 

grouped_rdd = mapped_rdd.groupByKey()

In [ ]:
# like this from 1000 machines we get the  customer id values for closed here  and we need to find their length 
# which means lot of shuffling is taking place here 
(closed,{11599,11502,39098,........}) # this will count the length of the the closed customer ids here which is nothing but total orders for closed 
(open,{........................}) # here all customer id are present for open 
(pending_payment,{..............}

In [5]:
# so we need to perform map transformation again 
result = grouped_rdd.map(lambda x:(x[0], len(x[1])))

In [ ]:
result.collect() # now we are executing the plan here 

In [ ]:
result.take(5)

In [ ]:
spark.stop()